<a href="https://colab.research.google.com/github/MihaiDogariu/CV3/blob/main/laborator/CV_3_Lab_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecția obiectelor cu modele pre-antrenate
Un detector de obiecte răspunde la întrebarea: ce obiecte sunt în imagine și unde se află fiecare? Pentru fiecare obiect găsit, detectorul întoarce o casetă de încadrare (bounding box), o clasă și un scor de încredere.

Antrenarea unui detector este costisitoare (zile de GPU pe sute de mii de imagini adnotate), dar există detectoare pre-antrenate pe baza de date MS-COCO (80 de clase de obiecte uzuale: persoane, vehicule, animale, bagaje, mobilier etc.), care pot fi folosite direct, fără nicio antrenare. În acest laborator utilizăm un detector pre-antrenat (off-the-shelf) și construim aplicații pe baza ieșirilor lui.

Concepte acoperite:
1. Formatul ieșirii unui detector (casete, clase, scoruri) și rolul pragului de încredere.
2. Aplicația 1 (detecția bagajelor abandonate): reguli simple, aplicate peste detecții, transformă un detector generic într-un sistem de supraveghere.
3. Aplicația 2 (detecția obiectelor într-un fișier video): un video este o secvență de imagini, deci detecția se reduce la 3 pași:
    1. video -> cadre
    2. detecție pe fiecare cadru
    3. cadre -> video.
4. Aplicația 3 (urmărirea obiectelor, tracking): asocierea detecțiilor din cadre consecutive, ca fiecare obiect să primească un identificator stabil. Construim un tracker în 3 pași, fiecare corectând o problemă vizibilă a pasului anterior, apoi îl folosim pentru a detecta bagajele abandonate în timp.

## 0. Configurare
- `MAX_FRAMES`: câte cadre se procesează din fiecare video. Pe GPU, 300 de cadre durează în jur de un minut pentru Faster R-CNN; pe CPU durează de zeci de ori mai mult, deci reduceți valoarea dacă rulați fără GPU. `None` procesează tot fișierul.
- `BATCH_SIZE`: câte cadre sunt procesate simultan de detector.

Pe lângă `torchvision`, folosim un pachet care nu este instalat implicit:
- `imageio-ffmpeg`: scrierea fișierelor video în format H.264, singurul format pe care browserele (și deci Colab/Jupyter) îl pot reda.

In [ ]:
import importlib.util
import collections
import itertools
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from IPython.display import Video, display
from PIL import Image
from scipy.optimize import linear_sum_assignment
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2
from torchvision.ops import box_iou, nms
from tqdm.auto import tqdm

if importlib.util.find_spec('imageio_ffmpeg') is None:
    %pip install -q imageio-ffmpeg
import imageio.v2 as imageio


# Alegem configurația sistemului (cpu/gpu)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else '')

DATA_DIR = './data/lab6'
OUT_DIR = os.path.join(DATA_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

MAX_FRAMES = 300
BATCH_SIZE = 4

### 0.1 Descărcarea resurselor
- 5 imagini cu bagaje, pentru aplicația 1;
- 3 secvențe video filmate de camere fixe, ca în supravegherea video.

In [ ]:
DRIVE_URL = 'https://drive.usercontent.google.com/download?id={}&export=download'
RESOURCES = {
    'img1.jpg': DRIVE_URL.format('1SrWV4NMGBy6cYhRvvlIjAxyKfqfIrWow'),
    'img2.jpg': DRIVE_URL.format('1bM3i5_ypub7XDJX4z8m7JO59XJAEVZdI'),
    'img3.jpg': DRIVE_URL.format('1kf8UJECW15tSesdgybeXAhjRBrju9ERH'),
    'img4.jpg': DRIVE_URL.format('1LhqNqYHjnxSB9e3O2TSEfV30kmk3-Qrp'),
    'img5.jpg': DRIVE_URL.format('11ZtCTGOtB_j99cjreaRgpszvXvkRdIPW'),
    'vtest.avi': 'https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/vtest.avi',
    'person-bicycle-car.mp4': 'https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4',
    'abandoned-bag.avi': 'https://raw.githubusercontent.com/harshagarwal10/abandoned-object-detection/master/video1.avi',
}

for name, url in RESOURCES.items():
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        torch.hub.download_url_to_file(url, path)
    print('{:<24} {:>6.1f} MB'.format(name, os.path.getsize(path) / 2**20))

IMAGE_PATHS = [os.path.join(DATA_DIR, 'img{}.jpg'.format(i)) for i in range(1, 6)]
# Lucrăm cu imagini RGB, ca tablouri numpy uint8 de dimensiune H x W x 3
images = [np.array(Image.open(path).convert('RGB')) for path in IMAGE_PATHS]

In [ ]:
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
frcnn = fasterrcnn_resnet50_fpn_v2(weights=weights).to(device).eval()
preprocess = weights.transforms()
COCO_CLASSES = weights.meta['categories']

print('Parametri: {:.1f} milioane'.format(sum(p.numel() for p in frcnn.parameters()) / 1e6))
print('mAP pe COCO val2017:', weights.meta['_metrics']['COCO-val2017']['box_map'])
print('Indici de clasă: {} (din care {} nefolosiți, marcați N/A)'.format(len(COCO_CLASSES), COCO_CLASSES.count('N/A')))
print(COCO_CLASSES)

Lista are 91 de poziții, dar doar 80 de clase reale: indexul 0 este fundalul (`__background__`), iar pozițiile `N/A` corespund unor clase planificate inițial în COCO și eliminate ulterior. Ele au fost păstrate pentru ca indicii claselor să rămână compatibili cu adnotările originale.

### 1.2 Funcția de detecție
Pre-procesarea din `weights.transforms()` doar transformă imaginea din `uint8` (0-255) în `float` (0-1). Restul (redimensionarea la latura mică de 800 de pixeli și normalizarea cu media și deviația standard ImageNet) se face în interiorul modelului. De aceea modelul acceptă o listă de imagini de dimensiuni diferite, nu un tensor cu formă fixă.

Pentru fiecare imagine, modelul întoarce un dicționar:
- `boxes`: tensor N x 4, cu coordonatele `(x1, y1, x2, y2)` ale fiecărei casete, în pixeli, în imaginea originală (colțul stânga-sus și colțul dreapta-jos);
- `labels`: N indici de clasă (poziții în `COCO_CLASSES`);
- `scores`: N scoruri de încredere, în ordine descrescătoare.

Modelul aplică deja NMS (non-maximum suppression): dintre casetele aceleiași clase care se suprapun puternic, o păstrează doar pe cea cu scorul cel mai mare. Noi setăm doar pragul de încredere.

`@torch.no_grad()` dezactivează calculul gradienților: la inferență nu avem nevoie de ei, iar fără ei detecția folosește mai puțină memorie și rulează mai repede.

In [ ]:
@torch.no_grad()
def detect_frcnn(images, threshold=0.5):
    # images: listă de imagini RGB (numpy uint8, H x W x 3)
    batch = [preprocess(torch.from_numpy(img).to(device).permute(2, 0, 1)) for img in images]  # H x W x 3 -> 3 x H x W, float în [0, 1]
    outputs = frcnn(batch)
    results = []
    for out in outputs:
        keep = out['scores'] >= threshold
        results.append(dict(
            boxes=out['boxes'][keep].cpu(),                                 # N x 4: (x1, y1, x2, y2)
            labels=[COCO_CLASSES[i] for i in out['labels'][keep].tolist()],
            scores=out['scores'][keep].cpu()))
    return results


det = detect_frcnn([images[1]])[0]
for box, label, score in zip(det['boxes'].round().int().tolist(), det['labels'], det['scores'].tolist()):
    print('{:<10} {:.3f}  {}'.format(label, score, box))

### 1.3 Vizualizarea detecțiilor și pragul de încredere
Desenăm casetele de încadrare cu OpenCV. Fiecare clasă primește o culoare fixă, iar grosimea liniilor și dimensiunea textului cresc cu dimensiunea imaginii, ca desenele să rămână vizibile și pe imaginile mari.

In [ ]:
PALETTE = (np.array(plt.cm.tab20.colors) * 255).astype(int).tolist()


def class_color(label):
    return tuple(PALETTE[COCO_CLASSES.index(label) % len(PALETTE)])


def draw_detections(image, det, colors=None):
    # colors: o culoare (r, g, b) pentru fiecare detecție; None = culoarea clasei; o detecție cu culoarea None nu se desenează
    img = image.copy()
    size = max(img.shape[:2])
    thickness = max(1, round(size / 500))
    font_scale = max(0.3, size / 1600)
    if colors is None:
        colors = [class_color(label) for label in det['labels']]
    for box, label, score, color in zip(det['boxes'].round().int().tolist(), det['labels'], det['scores'].tolist(), colors):
        if color is None:
            continue
        x1, y1, x2, y2 = box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
        cv2.putText(img, '{} {:.2f}'.format(label, score), (x1, max(y1 - 4, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, 1, cv2.LINE_AA)
    return img


def show_images(imgs, titles, ncols=3, width=6):
    nrows = (len(imgs) + ncols - 1) // ncols
    aspect = max(img.shape[0] / img.shape[1] for img in imgs)  # înălțimea fiecărui rând se potrivește cu cea mai înaltă imagine
    fig, axes = plt.subplots(nrows, ncols, figsize=(width * ncols, (width * aspect + 0.4) * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, img, title in zip(axes.flat, imgs, titles):
        ax.imshow(img)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

Pragul de încredere controlează compromisul dintre detecții ratate și detecții false. Aplicăm detectorul o singură dată cu un prag foarte mic, apoi filtrăm rezultatul pentru trei praguri diferite:

In [ ]:
def filter_detections(det, threshold):
    keep = det['scores'] >= threshold
    return dict(boxes=det['boxes'][keep], labels=[l for l, k in zip(det['labels'], keep) if k], scores=det['scores'][keep])


det = detect_frcnn([images[4]], threshold=0.05)[0]
thresholds = [0.1, 0.5, 0.9]
show_images([draw_detections(images[4], filter_detections(det, t)) for t in thresholds],
            ['prag {}: {} detecții'.format(t, (det['scores'] >= t).sum().item()) for t in thresholds])

- La pragul 0,1 apar zeci de casete: multe sunt duplicate parțiale, părți de obiecte sau obiecte inexistente.
- La pragul 0,9 rămân doar obiectele foarte clare, dar se pierd și detecții corecte.
- Pragul 0,5 este un compromis rezonabil și îl folosim în continuare pentru Faster R-CNN. Valoarea potrivită depinde însă de aplicație (cât costă un fals pozitiv față de un fals negativ?).

## 2. Aplicația 1: detecția bagajelor abandonate
Implementăm un sistem simplificat, care emite o alarmă atunci când în imagine apare un bagaj abandonat.

- Bagaj: orice obiect din clasele `backpack`, `handbag` sau `suitcase`.
- Bagaj abandonat: un bagaj care nu se află în posesia unei persoane. Considerăm că un bagaj se află în posesia unei persoane atunci când caseta bagajului se intersectează cu caseta a cel puțin unei persoane.

Afișare: chenar roșu pentru bagajele abandonate, verde pentru bagajele însoțite, albastru pentru persoane. Celelalte detecții nu sunt relevante pentru aplicație și nu se desenează.

### 2.1 Regula de posesie
Intersecția a două casete are arie nenulă dacă și numai dacă IoU (intersection over union) este strict pozitiv, deci putem folosi direct `torchvision.ops.box_iou`. Funcția primește `n` casete de bagaje și `m` casete de persoane și întoarce matricea `n x m` a valorilor IoU, calculată vectorial, fără bucle Python. Un bagaj este însoțit dacă are cel puțin o valoare nenulă pe rândul său.

Parametrul `margin` va fi folosit în secțiunea 2.3; pentru `margin=0` obținem exact regula de mai sus. Cazurile limită se tratează natural: dacă în imagine nu există nicio persoană, matricea are 0 coloane, `any` întoarce `False` pentru fiecare bagaj și toate bagajele sunt abandonate.

In [ ]:
LUGGAGE = {'backpack', 'handbag', 'suitcase'}
STATUS_COLORS = {'persoana': (40, 120, 255), 'insotit': (30, 200, 30), 'abandonat': (230, 30, 30)}


def find_abandoned(det, margin=0.0):
    # întoarce statusul fiecărei detecții: 'persoana', 'insotit', 'abandonat' sau None (alte clase)
    is_person = torch.tensor([label == 'person' for label in det['labels']], dtype=torch.bool)
    is_bag = torch.tensor([label in LUGGAGE for label in det['labels']], dtype=torch.bool)
    persons = det['boxes'][is_person]
    bags = det['boxes'][is_bag]

    # Extindem caseta fiecărei persoane cu margin x înălțimea ei, în toate direcțiile
    pad = margin * (persons[:, 3] - persons[:, 1])
    persons = persons + torch.stack([-pad, -pad, pad, pad], dim=1)

    attended = (box_iou(bags, persons) > 0).any(dim=1)  # n x m -> n
    bag_status = iter(['insotit' if a else 'abandonat' for a in attended.tolist()])
    return ['persoana' if p else next(bag_status) if b else None for p, b in zip(is_person.tolist(), is_bag.tolist())]


def luggage_alarm(image, det, margin=0.0, name=''):
    status = find_abandoned(det, margin)
    n_abandoned = status.count('abandonat')
    if n_abandoned > 0:
        print('ALARMĂ {}: {} bagaj(e) abandonat(e)'.format(name, n_abandoned))
    img = draw_detections(image, det, colors=[STATUS_COLORS.get(s) for s in status])
    title = '{}: {} însoțite, {} abandonate'.format(name, status.count('insotit'), n_abandoned)
    return img, title

### 2.2 Rezultate
Rulăm detectorul pe toate cele 5 imagini într-un singur batch, apoi aplicăm regula.

In [ ]:
NAMES = [os.path.basename(path) for path in IMAGE_PATHS]
frcnn_dets = detect_frcnn(images, threshold=0.5)
results = [luggage_alarm(img, det, name=name) for img, det, name in zip(images, frcnn_dets, NAMES)]
show_images(*zip(*results))

- `img1`: valiza singură pe hol este semnalată corect, iar imaginea fără nicio persoană declanșează alarma.
- `img2`: valiza trasă de persoană se intersectează cu ea, deci este însoțită.
- `img3`: nu există bagaje, deci nu există alarmă. Imaginile fără evenimente sunt la fel de importante ca cele cu evenimente: un sistem de supraveghere rulează aproape tot timpul pe astfel de imagini, iar fiecare alarmă falsă scade încrederea operatorului în sistem.
- `img4`: doar rucsacul de lângă copil este însoțit; celelalte 5 declanșează alarma, deși aparțin, evident, copilului. Analizăm cazul în secțiunea 2.3.
- `img5`: ambele valize se ating de persoană, deci nu există alarmă. Valiza galbenă are însă două casete (scoruri 0,95 și 0,76): a doua acoperă doar partea de sus a valizei, iar suprapunerea dintre ele este prea mică pentru ca NMS să o elimine. Un sistem care numără bagajele ar raporta aici 3 în loc de 2.

### 2.3 Limitele regulii
Regula de intersecție lucrează în planul imaginii, nu în lumea reală, iar acest lucru produce ambele tipuri de erori:
- alarme false: un bagaj așezat lângă proprietar, dar fără ca acesta să se atingă în imagine (rucsacele din `img4`, care stau în șirul copilului) este considerat abandonat;
- bagaje abandonate ratate: din cauza perspectivei, o persoană care trece prin fața unui bagaj abandonat, la câțiva metri de el, se intersectează cu el în imagine și îl face să pară însoțit.

O primă încercare de a reduce alarmele false este să tolerăm o distanță mică între bagaj și persoană. Extindem caseta fiecărei persoane cu o fracțiune din înălțimea ei. Raportarea la înălțimea persoanei, și nu la un număr fix de pixeli, face regula independentă de distanța față de cameră: o persoană aflată departe este mică în imagine, deci și distanța tolerată devine mică.

In [ ]:
margins = [0.0, 0.25, 1.0]
results = [luggage_alarm(images[3], frcnn_dets[3], margin=m, name='margin={}'.format(m)) for m in margins]
show_images(*zip(*results))

- Cu `margin=0.25` (un sfert din înălțimea copilului), devine însoțit doar rucsacul vecin. Celelalte sunt prea departe de copil pentru orice margine rezonabilă.
- Cu `margin=1.0`, caseta copilului se extinde cu încă o înălțime a lui în fiecare direcție, iar 4 din cele 6 bagaje devin însoțite. Problema este că acum ar fi considerat însoțit și un bagaj aflat la doi metri de un trecător oarecare.

Rucsacele formează un șir care începe lângă copil: intuitiv, îi aparțin toate, dar niciun criteriu de distanță față de persoană nu poate exprima acest lucru (vezi întrebarea 1 din #TODO). Alegerea marginii rămâne un compromis între alarme false și evenimente ratate, la fel ca alegerea pragului de încredere.

Două observații mai importante decât valoarea marginii:
1. Regula ignoră timpul. O imagine statică nu poate distinge un bagaj abandonat de unul lăsat jos pentru câteva secunde. Un sistem real urmărește bagajul și proprietarul pe parcursul mai multor cadre și declanșează alarma doar dacă bagajul rămâne nesupravegheat mai mult de un anumit timp (de exemplu, 30 de secunde, în protocolul PETS 2006). Pentru asta avem nevoie de video și de urmărirea obiectelor, adică de aplicațiile 2 și 3.
2. Regula depinde de calitatea detectorului. În `img4`, unul dintre rucsaci este detectat ca `kite` (zmeu), deci nu este considerat bagaj deloc. Erorile detectorului se propagă direct în decizia sistemului.

## 3. Aplicația 2: detecția obiectelor într-un fișier video
Un fișier video este o secvență de imagini (cadre) afișate la o frecvență fixă (FPS, frames per second). Detectorul nu are nicio noțiune de timp: îl aplicăm independent pe fiecare cadru. Procesul are 3 pași:
1. transformarea fișierului video în secvență de cadre;
2. rularea detecției pe fiecare cadru;
3. transformarea secvenței de cadre prelucrate înapoi într-un fișier video și afișarea lui.

### 3.1 Citirea cadrelor
`cv2.VideoCapture` decodează fișierul cadru cu cadru. Două detalii practice:
- OpenCV întoarce cadrele în ordinea de canale BGR, din motive istorice, iar detectorul a fost antrenat pe imagini RGB. Uitarea conversiei nu produce nicio eroare, dar degradează detecția (de exemplu, cerul devine portocaliu).
- Nu încărcăm tot videoclipul în memorie: 300 de cadre de 768 x 576 ocupă deja aproape 400 MB. Funcția `read_frames` este un generator (`yield`): produce câte un cadru doar atunci când este cerut.

Parametrii `start` și `step` permit procesarea unei porțiuni din video (de la cadrul `start`) și a unui cadru din `step`. Îi vom folosi în secțiunea 4.4, pentru un videoclip lung, filmat la 30 de cadre pe secundă.

In [ ]:
def video_info(path):
    cap = cv2.VideoCapture(path)
    info = dict(fps=cap.get(cv2.CAP_PROP_FPS), frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()
    return info


def read_frames(path, max_frames=None, start=0, step=1):
    cap = cv2.VideoCapture(path)
    index, count = 0, 0
    while max_frames is None or count < max_frames:
        ok, frame = cap.read()
        if not ok:  # sfârșitul fișierului
            break
        if index >= start and (index - start) % step == 0:
            yield cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            count += 1
        index += 1
    cap.release()


VIDEO_1 = os.path.join(DATA_DIR, 'person-bicycle-car.mp4')
info = video_info(VIDEO_1)
print(info, '-> {:.1f} secunde'.format(info['frames'] / info['fps']))

### 3.2 Detecția pe fiecare cadru
`itertools.islice` extrage din generator următoarele `batch_size` cadre, care trec împreună prin detector.

Măsurăm și viteza detectorului, în cadre pe secundă. Două precauții:
- primul apel al unui model pe GPU este mai lent (se alocă memoria, se aleg algoritmii de convoluție potriviți), deci îl facem înainte de măsurare, pe un singur cadru (warm-up);
- operațiile pe GPU sunt asincrone: Python poate trece mai departe înainte ca GPU-ul să termine calculul. `torch.cuda.synchronize()` așteaptă finalizarea lor, ca timpul măsurat să fie cel real.

Măsurăm doar timpul detecției, fără decodarea video: aceasta este partea care depinde de model.

In [ ]:
def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()


def detect_video(path, detect_fn, threshold, max_frames=MAX_FRAMES, batch_size=BATCH_SIZE, start=0, step=1):
    frames = read_frames(path, max_frames, start, step)
    total = min(max_frames or float('inf'), (video_info(path)['frames'] - start + step - 1) // step)
    detect_fn([next(read_frames(path))], threshold)  # warm-up
    detections, elapsed = [], 0.0
    with tqdm(total=total, desc=os.path.basename(path)) as progress:
        while True:
            batch = list(itertools.islice(frames, batch_size))
            if not batch:  # generatorul nu mai are cadre
                break
            synchronize()
            start = time.perf_counter()
            detections += detect_fn(batch, threshold)
            synchronize()
            elapsed += time.perf_counter() - start
            progress.update(len(batch))
    fps = len(detections) / elapsed
    print('{} cadre, {:.1f} cadre/secundă'.format(len(detections), fps))
    return detections, fps


# Procesăm tot fișierul: bicicleta și mașina apar abia în a doua jumătate a videoclipului
frcnn_video_1, _ = detect_video(VIDEO_1, detect_frcnn, threshold=0.5, max_frames=None)

### 3.3 Reconstrucția videoclipului și afișarea
Citim din nou cadrele, desenăm detecțiile salvate și scriem cadrele într-un fișier nou, cu aceeași frecvență ca originalul.

Folosim `imageio` și nu `cv2.VideoWriter`, deoarece codecurile disponibile în OpenCV (de exemplu, `mp4v`) produc fișiere pe care browserele nu le pot reda, deci videoclipul nu ar putea fi afișat în notebook. `imageio` folosește `ffmpeg` cu codecul H.264 (`libx264`), suportat de toate browserele. `Video(..., embed=True)` include apoi fișierul direct în notebook.

Funcția acceptă mai multe seturi de rezultate pentru același video și le așază unul lângă altul. Vom folosi acest lucru în secțiunea 4, pentru a compara două variante de tracker. Parametrul `draw_fn` alege funcția de desenare: în secțiunea 4 vom desena traiectorii, nu detecții.

In [ ]:
def render_video(path, out_path, all_detections, titles=None, draw_fn=draw_detections, start=0, step=1):
    # all_detections: o listă de rezultate per cadru pentru fiecare model; fiecare model ocupă un panou
    fps = video_info(path)['fps'] / step
    with imageio.get_writer(out_path, fps=fps, codec='libx264', macro_block_size=16) as writer:
        for i, frame in enumerate(read_frames(path, len(all_detections[0]), start, step)):
            panels = [draw_fn(frame, detections[i]) for detections in all_detections]
            if titles:
                for panel, title in zip(panels, titles):
                    cv2.putText(panel, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
            writer.append_data(np.hstack(panels))
    return out_path


out_1 = render_video(VIDEO_1, os.path.join(OUT_DIR, 'person-bicycle-car_frcnn.mp4'), [frcnn_video_1])
display(Video(out_1, embed=True, width=768))

Câteva cadre, pentru o privire mai atentă (și pentru cazul în care videoclipul nu poate fi redat):

In [ ]:
def show_video_frames(path, all_detections, frame_ids, titles=None, draw_fn=draw_detections, start=0, step=1):
    frames = list(read_frames(path, max(frame_ids) + 1, start, step))
    imgs = [np.hstack([draw_fn(frames[i], detections[i]) for detections in all_detections]) for i in frame_ids]
    show_images(imgs, ['cadrul {}'.format(i) + (': ' + ' | '.join(titles) if titles else '') for i in frame_ids],
                ncols=1 if len(all_detections) > 1 else 3, width=6 * len(all_detections))


show_video_frames(VIDEO_1, [frcnn_video_1], [60, 330, 560])

Detecțiile sunt calculate independent pe fiecare cadru, deci pot apărea și dispărea de la un cadru la altul (de exemplu, când un obiect este parțial acoperit), iar casetele "tremură" ușor. Detectorul nu știe că persoana din cadrul 60 este aceeași cu cea din cadrul 61: pentru asta este nevoie de un algoritm de urmărire (tracking), care asociază detecțiile din cadre consecutive. Tracking-ul este și piesa care lipsește pentru a aplica regula de bagaj abandonat în timp (secțiunea 2.3).

## 4. Aplicația 3: urmărirea obiectelor (tracking)
Resurse utile:
- SORT: A. Bewley et al., Simple Online and Realtime Tracking, ICIP 2016 (https://arxiv.org/abs/1602.00763)
- algoritmul ungar: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html

În secțiunea 3, detectorul a lucrat independent pe fiecare cadru: știe că în cadrul 61 există 7 persoane, dar nu știe care dintre ele este persoana din cadrul 60. Un tracker rezolvă această problemă. El asociază detecțiile din cadre consecutive și atribuie fiecărui obiect un identificator (ID) care rămâne același cât timp obiectul este în scenă. Succesiunea de casete cu același ID formează o traiectorie (track).

Abordarea pe care o folosim se numește tracking-by-detection: detectorul rămâne neschimbat, iar tracker-ul este un pas de post-procesare, aplicat peste detecțiile deja calculate. Tracker-ul este și online: decizia pentru cadrul `t` folosește doar cadrele `0 ... t`, deci poate rula în timp real, pe măsură ce sosesc cadrele.

Construim tracker-ul în 3 pași. Fiecare pas corectează o problemă pe care o vom vedea concret la pasul anterior:
1. asocierea după IoU: o detecție continuă traiectoria cu care se suprapune cel mai mult;
2. ciclul de viață al traiectoriilor: traiectoriile supraviețuiesc câteva cadre fără detecții, iar cele noi sunt confirmate abia după câteva cadre;
3. predicția mișcării și asocierea optimă: căutăm obiectul acolo unde va fi, nu acolo unde a fost, și alegem asocierile global, nu pe rând.

Rezultatul este o versiune simplificată a tracker-ului SORT. Lucrăm pe `vtest.avi`: o scenă cu mulți pietoni, filmați de la distanță, care se intersectează și sunt acoperiți uneori de stâlpi și panouri. Rulăm detectorul o singură dată, pe toate cadrele; tracker-ul lucrează apoi doar pe detecțiile salvate, deci îl putem rula de mai multe ori fără a repeta detecția.

In [ ]:
VIDEO_2 = os.path.join(DATA_DIR, 'vtest.avi')
print(video_info(VIDEO_2))
frcnn_video_2, _ = detect_video(VIDEO_2, detect_frcnn, threshold=0.5)

### 4.1 Pasul 1: asocierea după IoU
Între două cadre consecutive, un pieton se deplasează doar câțiva pixeli (`vtest.avi` are 10 cadre pe secundă), deci caseta lui din cadrul `t` se suprapune puternic cu caseta din cadrul `t-1`. Algoritmul, pentru fiecare cadru nou:
1. calculează matricea IoU dintre ultimele casete ale traiectoriilor existente (rânduri) și detecțiile noi (coloane);
2. asociere greedy: alege perechea cu IoU maxim, o asociază, elimină rândul și coloana ei din matrice și repetă, cât timp IoU maxim depășește un prag (0,3);
3. o detecție neasociată pornește o traiectorie nouă, cu un ID nou; o traiectorie neasociată este ștearsă.

Două detalii:
- nu asociem obiecte din categorii diferite: IoU este înmulțit cu 0 pentru perechile persoană - bagaj;
- toate bagajele formează o singură categorie, pentru că detectorul poate eticheta același rucsac drept `backpack` într-un cadru și `handbag` în următorul (vom vedea acest lucru în secțiunea 4.4).

Clasa `Track` conține deja și câmpurile folosite în pașii 2 și 3; pasul 1 folosește doar `id`, `box` și `label`.

In [ ]:
def category(label):
    return 'bagaj' if label in LUGGAGE else label


def iou_matrix(track_boxes, track_labels, det):
    # rând = traiectorie, coloană = detecție; perechile din categorii diferite primesc IoU = 0
    iou = box_iou(track_boxes, det['boxes'])
    same = torch.tensor([[category(a) == category(b) for b in det['labels']] for a in track_labels], dtype=torch.bool)
    return iou * same.view(iou.shape)


def match_greedy(iou, threshold):
    iou = iou.clone()
    matches = []
    # TODO: cât timp matricea nu este goală și maximul ei este >= threshold: găsiți poziția (t, d) a maximului,
    # adăugați perechea în matches și excludeți rândul t și coloana d din căutările următoare (secțiunea 4.1)

    return matches


class Track:
    def __init__(self, track_id, box, label):
        self.id = track_id
        self.box = box                  # ultima casetă asociată
        self.label = label
        self.hits = 1                   # pasul 2: în câte cadre a fost asociată
        self.misses = 0                 # pasul 2: de câte cadre consecutive nu a mai fost asociată
        self.velocity = torch.zeros(4)  # pasul 3: deplasarea medie a casetei într-un cadru


class IoUTracker:
    def __init__(self, iou_threshold=0.3):
        self.iou_threshold = iou_threshold
        self.tracks = []
        self.next_id = 1

    def update(self, det):
        boxes = torch.stack([t.box for t in self.tracks]) if self.tracks else torch.zeros(0, 4)
        matches = match_greedy(iou_matrix(boxes, [t.label for t in self.tracks], det), self.iou_threshold)

        for t, d in matches:
            self.tracks[t].box = det['boxes'][d]
        matched_tracks = {t for t, _ in matches}
        self.tracks = [track for t, track in enumerate(self.tracks) if t in matched_tracks]  # traiectoriile neasociate se șterg

        matched_dets = {d for _, d in matches}
        for d in range(len(det['boxes'])):
            if d not in matched_dets:  # detecție neasociată -> traiectorie nouă
                self.tracks.append(Track(self.next_id, det['boxes'][d], det['labels'][d]))
                self.next_id += 1
        return self.tracks

Funcțiile de mai jos sunt aceleași pentru toți pașii:
- `run_tracker` aplică tracker-ul cadru cu cadru și păstrează, pentru desenare, ultimele 30 de poziții ale fiecărui ID (urma, trail): mijlocul laturii de jos a casetei, adică poziția picioarelor;
- `track_stats` rezumă rezultatul. Nu avem adnotări, deci folosim trei indicatori simpli: numărul de ID-uri distincte (ideal, egal cu numărul real de persoane), lungimea medie a unei traiectorii și numărul de fragmente, traiectorii mai scurte de 10 cadre (o secundă), care sunt aproape întotdeauna bucăți din traiectoria cuiva care și-a pierdut ID-ul;
- `draw_tracks` desenează fiecare ID cu o culoare proprie și urma lui.

In [ ]:
TRAIL = 30


def select_labels(det, labels):
    keep = torch.tensor([label in labels for label in det['labels']], dtype=torch.bool)
    return dict(boxes=det['boxes'][keep], labels=[l for l, k in zip(det['labels'], keep) if k], scores=det['scores'][keep])


def bottom_center(box):
    return (int((box[0] + box[2]) / 2), int(box[3]))


def run_tracker(tracker, detections, labels=('person',)):
    outputs, trails = [], collections.defaultdict(list)
    for det in detections:
        tracks = tracker.update(select_labels(det, labels))
        frame_tracks = []
        for t in tracks:
            trails[t.id] = (trails[t.id] + [bottom_center(t.box)])[-TRAIL:]
            frame_tracks.append(dict(id=t.id, box=t.box, label=t.label, trail=trails[t.id]))
        outputs.append(frame_tracks)
    return outputs


def track_stats(outputs, name):
    lengths = np.array(list(collections.Counter(t['id'] for frame in outputs for t in frame).values()))
    print('{:<34} ID-uri: {:>3} | lungime medie: {:>5.1f} cadre | fragmente (< 10 cadre): {:>3}'.format(
        name, len(lengths), lengths.mean(), (lengths < 10).sum()))


def draw_tracks(image, tracks):
    img = image.copy()
    for t in tracks:
        color = t.get('color') or tuple(PALETTE[t['id'] % len(PALETTE)])
        x1, y1, x2, y2 = t['box'].round().int().tolist()
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, t.get('text', '#{}'.format(t['id'])), (x1, max(y1 - 4, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        if len(t.get('trail', [])) > 1:
            cv2.polylines(img, [np.array(t['trail'], dtype=np.int32)], False, color, 2)
    return img


tracks_1 = run_tracker(IoUTracker(), frcnn_video_2)
track_stats(tracks_1, 'Pasul 1: IoU greedy')

Tracker-ul a creat mult mai multe ID-uri decât persoane vizibile în videoclip, iar majoritatea sunt fragmente. Graficul de mai jos arată, pentru fiecare ID (pe verticală), cadrele în care traiectoria lui este activă. O persoană urmărită corect ar apărea ca un singur segment lung; aici, multe persoane sunt rupte în segmente scurte, unul după altul.

In [ ]:
def plot_track_timeline(all_outputs, titles):
    fig, axes = plt.subplots(1, len(all_outputs), figsize=(6 * len(all_outputs), 4), squeeze=False)
    for ax, outputs, title in zip(axes[0], all_outputs, titles):
        frames_of = collections.defaultdict(list)
        for f, frame in enumerate(outputs):
            for t in frame:
                frames_of[t['id']].append(f)
        for row, (track_id, frames) in enumerate(sorted(frames_of.items(), key=lambda item: item[1][0])):
            ax.scatter(frames, [row] * len(frames), s=1, color=np.array(PALETTE[track_id % len(PALETTE)]) / 255)
        ax.set_title('{}: {} ID-uri'.format(title, len(frames_of)))
        ax.set_xlabel('cadru')
        ax.set_ylabel('ID (ordonate după apariție)')
    plt.tight_layout()
    plt.show()


plot_track_timeline([tracks_1], ['Pasul 1'])

Cauza este vizibilă direct în algoritm: o traiectorie neasociată este ștearsă imediat. Este suficient ca detectorul să rateze o persoană într-un singur cadru (o ocluzie de către stâlp sau panou, o postură neobișnuită, un scor puțin sub prag) pentru ca, în cadrul următor, aceeași persoană să primească un ID nou. În sens invers, fiecare detecție falsă izolată devine o traiectorie de un cadru.

### 4.2 Pasul 2: ciclul de viață al traiectoriilor
Două reguli simple, controlate de câte un parametru:
- `max_age`: o traiectorie neasociată nu este ștearsă imediat, ci păstrată până la `max_age` cadre consecutive fără detecții (contorul `misses`). Dacă obiectul reapare în acest interval, lângă ultima poziție cunoscută, își recuperează ID-ul.
- `min_hits`: o traiectorie nouă este afișată abia după ce a fost asociată în `min_hits` cadre (contorul `hits`). O detecție falsă izolată nu ajunge, astfel, să producă un ID.

Tracker-ul întoarce doar traiectoriile confirmate și asociate în cadrul curent; cele aflate "în așteptare" (`misses > 0`) rămân în memorie, dar nu se desenează. Liniile noi față de pasul 1 sunt marcate cu `# nou`.

In [ ]:
class LifecycleTracker:
    def __init__(self, iou_threshold=0.3, max_age=10, min_hits=3):
        self.iou_threshold = iou_threshold
        self.max_age = max_age    # nou
        self.min_hits = min_hits  # nou
        self.tracks = []
        self.next_id = 1

    def update(self, det):
        boxes = torch.stack([t.box for t in self.tracks]) if self.tracks else torch.zeros(0, 4)
        matches = match_greedy(iou_matrix(boxes, [t.label for t in self.tracks], det), self.iou_threshold)

        # TODO (nou): actualizați contoarele misses și hits ale traiectoriilor (secțiunea 4.2)
        for t, d in matches:
            self.tracks[t].box = det['boxes'][d]

        # TODO (nou): ștergeți doar traiectoriile care nu au fost asociate de mai mult de max_age cadre consecutive

        matched_dets = {d for _, d in matches}
        for d in range(len(det['boxes'])):
            if d not in matched_dets:
                self.tracks.append(Track(self.next_id, det['boxes'][d], det['labels'][d]))
                self.next_id += 1
        # TODO (nou): întoarceți doar traiectoriile confirmate și asociate în cadrul curent
        return self.tracks


tracks_2 = run_tracker(LifecycleTracker(), frcnn_video_2)
track_stats(tracks_1, 'Pasul 1: IoU greedy')
track_stats(tracks_2, 'Pasul 2: + max_age, min_hits')

Numărul de ID-uri scade aproximativ la jumătate. `max_age = 10` cadre înseamnă o secundă la 10 cadre pe secundă: suficient pentru ocluzii scurte, dar nu atât de mult încât o traiectorie "moartă" să fure detecțiile altui obiect care trece prin același loc.

Rămâne o problemă: în timpul unei ocluzii, persoana continuă să meargă, dar caseta traiectoriei rămâne pe loc, la ultima poziție cunoscută. Cu cât ocluzia durează mai mult, cu atât persoana reapare mai departe de această casetă, iar IoU poate scădea sub prag.

### 4.3 Pasul 3: predicția mișcării și asocierea optimă
Predicția. Fiecare traiectorie își estimează viteza: deplasarea medie a casetei (toate cele 4 coordonate) într-un cadru. Înainte de asociere, caseta este mutată cu `velocity x misses`, adică acolo unde ar trebui să fie obiectul dacă a continuat să se miște la fel. Viteza este netezită exponențial, `v = 0.7 v + 0.3 deplasare`, ca o singură casetă imprecisă să nu schimbe brusc direcția estimată.

SORT folosește aici un filtru Kalman: aceeași idee (model de mișcare cu viteză constantă, predicție, corecție la fiecare măsurătoare), dar cu o estimare explicită a incertitudinii, care decide automat cât de mult se crede predicția și cât de mult se crede detecția. Netezirea exponențială este o versiune simplificată, cu o pondere fixă (0,7).

Asocierea optimă. Asocierea greedy alege la fiecare pas perechea cu IoU maxim, chiar dacă alegerea strică o asociere mai bună în altă parte. Algoritmul ungar (`linear_sum_assignment`) alege setul de perechi cu suma IoU maximă.

In [ ]:
def match_hungarian(iou, threshold):
    rows, cols = linear_sum_assignment(iou.numpy(), maximize=True)
    return [(t, d) for t, d in zip(rows.tolist(), cols.tolist()) if iou[t, d] >= threshold]

In [ ]:
class MotionTracker:
    def __init__(self, iou_threshold=0.3, max_age=10, min_hits=3, smoothing=0.7, matcher=match_hungarian):
        self.iou_threshold = iou_threshold
        self.max_age = max_age
        self.min_hits = min_hits
        self.smoothing = smoothing  # nou
        self.matcher = matcher      # nou
        self.tracks = []
        self.next_id = 1

    def update(self, det):
        for track in self.tracks:
            track.misses += 1
        # TODO (nou): calculați predicted, casetele prezise ale traiectoriilor (tensor T x 4), conform secțiunii 4.3.
        # Atenție la cazul în care nu există nicio traiectorie

        matches = self.matcher(iou_matrix(predicted, [t.label for t in self.tracks], det), self.iou_threshold)

        for t, d in matches:
            track = self.tracks[t]
            # TODO (nou): actualizați track.velocity cu deplasarea pe cadru, netezită exponențial cu self.smoothing.
            # Atenție: între ultima asociere și cea curentă au trecut track.misses cadre

            track.box = det['boxes'][d]
            track.misses, track.hits = 0, track.hits + 1
        self.tracks = [t for t in self.tracks if t.misses <= self.max_age]

        matched_dets = {d for _, d in matches}
        for d in range(len(det['boxes'])):
            if d not in matched_dets:
                self.tracks.append(Track(self.next_id, det['boxes'][d], det['labels'][d]))
                self.next_id += 1
        return [t for t in self.tracks if t.misses == 0 and t.hits >= self.min_hits]


tracks_3 = run_tracker(MotionTracker(), frcnn_video_2)
track_stats(tracks_1, 'Pasul 1: IoU greedy')
track_stats(tracks_2, 'Pasul 2: + max_age, min_hits')
track_stats(run_tracker(MotionTracker(matcher=match_greedy), frcnn_video_2), 'Pasul 3, doar predicția')
track_stats(tracks_3, 'Pasul 3: + predicție, alg. ungar')
plot_track_timeline([tracks_1, tracks_2, tracks_3], ['Pasul 1', 'Pasul 2', 'Pasul 3'])

- Predicția mișcării aduce încă o scădere a numărului de ID-uri și traiectorii mai lungi.
- Algoritmul ungar nu schimbă aproape nimic pe această secvență: persoanele sunt, în general, suficient de departe una de alta încât să nu "concureze" pentru aceeași detecție. Diferența apare în scene aglomerate. Este un exemplu util: o metodă corectă în principiu nu îmbunătățește neapărat rezultatul pe orice date.

Cele trei variante, cadru cu cadru: în stânga pasul 1, la mijloc pasul 2, în dreapta pasul 3. Urmăriți culorile: în stânga, aceeași persoană își schimbă frecvent culoarea (deci ID-ul), iar urmele sunt scurte.

In [ ]:
TITLES_TRACK = ['Pasul 1: IoU greedy', 'Pasul 2: + max_age, min_hits', 'Pasul 3: SORT simplificat']
out_t = render_video(VIDEO_2, os.path.join(OUT_DIR, 'vtest_tracking.mp4'), [tracks_1, tracks_2, tracks_3], TITLES_TRACK, draw_fn=draw_tracks)
display(Video(out_t, embed=True, width=1500))
show_video_frames(VIDEO_2, [tracks_1, tracks_2, tracks_3], [120, 200], TITLES_TRACK, draw_fn=draw_tracks)

### 4.4 Bagaje abandonate în timp
În secțiunea 2.3 am concluzionat că o imagine statică nu poate distinge un bagaj abandonat de unul lăsat jos pentru câteva secunde. Cu tracking, regula poate include timpul:

> un bagaj este abandonat dacă nicio persoană nu se află lângă el (regula din secțiunea 2, cu `margin = 0.25`) timp de cel puțin `T` secunde consecutive.

Tracking-ul este necesar pentru că timpul se măsoară per bagaj: avem nevoie să știm că rucsacul din cadrul curent este același cu cel din cadrul anterior, ca să continuăm să îi numărăm secundele.

Folosim videoclipul `abandoned-bag.avi` (640 x 480, 30 de cadre pe secundă, 73 de secunde): două persoane intră cu un rucsac, discută, lasă rucsacul pe jos și pleacă împreună (la aproximativ 47 de secunde); după aceea, alte persoane trec prin hol, fără să se apropie de rucsac. Pentru a reduce timpul de procesare, începem de la secunda 30 (cadrul 900) și păstrăm un cadru din 2 (15 cadre pe secundă, suficient pentru tracking).

In [ ]:
VIDEO_3 = os.path.join(DATA_DIR, 'abandoned-bag.avi')
BAG_START, BAG_STEP = 900, 2
info_3 = video_info(VIDEO_3)
BAG_FPS = info_3['fps'] / BAG_STEP
print(info_3, '-> procesăm {:.0f} cadre pe secundă'.format(BAG_FPS))

bag_dets, _ = detect_video(VIDEO_3, detect_frcnn, threshold=0.5, max_frames=None, start=BAG_START, step=BAG_STEP)


def frame_time(f):
    # timpul (în secunde, în videoclipul original) al cadrului procesat f
    return (BAG_START + f * BAG_STEP) / info_3['fps']

Detecții duble. După ce este lăsat jos, rucsacul este detectat de două ori: o dată ca `handbag` și o dată ca `backpack`, cu casete aproape identice. NMS-ul din interiorul detectorului (secțiunea 1.2) compară doar casete din aceeași clasă, deci nu le poate elimina. Pentru aplicația noastră, clasa exactă a bagajului nu contează, așa că aplicăm încă o dată NMS, pe toate bagajele împreună (`torchvision.ops.nms`), și păstrăm doar caseta cu scorul cel mai mare.

In [ ]:
def merge_luggage(det, iou_threshold=0.5):
    is_bag = torch.tensor([label in LUGGAGE for label in det['labels']], dtype=torch.bool)
    bag_idx = torch.where(is_bag)[0]
    bag_keep = bag_idx[nms(det['boxes'][bag_idx], det['scores'][bag_idx], iou_threshold)]  # indicii rămași, ordonați după scor
    keep = torch.cat([torch.where(~is_bag)[0], bag_keep])
    return dict(boxes=det['boxes'][keep], labels=[det['labels'][i] for i in keep.tolist()], scores=det['scores'][keep])


count_bags = lambda dets: np.mean([sum(label in LUGGAGE for label in det['labels']) for det in dets])
print('Bagaje pe cadru: {:.2f} înainte, {:.2f} după NMS pe toate bagajele'.format(count_bags(bag_dets), count_bags([merge_luggage(d) for d in bag_dets])))

Monitorizarea. La fiecare cadru:
1. tracker-ul primește persoanele și bagajele (după `merge_luggage`);
2. pentru fiecare bagaj urmărit aplicăm `find_abandoned` din secțiunea 2.1, față de persoanele urmărite în acel cadru;
3. `unattended_since[id]` reține cadrul din care bagajul este nesupravegheat; se șterge când o persoană se apropie de bagaj, iar când diferența depășește `T` secunde se declanșează alarma.

Un detaliu important: verificăm toate bagajele aflate în memoria tracker-ului (`tracker.tracks`), nu doar cele detectate în cadrul curent. Un bagaj nedetectat într-un cadru (`misses > 0`), dar încă urmărit, își păstrează cronometrul. Acesta este exact avantajul pașilor 2 și 3.

Culori: albastru pentru persoane, verde pentru bagajele supravegheate, portocaliu pentru bagajele nesupravegheate de mai puțin de `T` secunde (cu cronometrul afișat) și roșu pentru bagajele abandonate.

In [ ]:
ALARM_AFTER = 3.0  # secunde; protocolul PETS 2006 folosește 30 de secunde, aici scurtăm pentru un videoclip scurt
MONITOR_COLORS = dict(persoana=(40, 120, 255), insotit=(30, 200, 30), nesupravegheat=(255, 150, 0), abandonat=(230, 30, 30))


def monitor_luggage(tracker, detections, fps, alarm_after=ALARM_AFTER, margin=0.25):
    unattended_since = {}  # id bagaj -> cadrul din care nicio persoană nu se află lângă el
    outputs, timers = [], []
    for f, det in enumerate(detections):
        shown = tracker.update(merge_luggage(select_labels(det, LUGGAGE | {'person'})))
        persons = [t for t in shown if t.label == 'person']
        bags = [t for t in tracker.tracks if t.label in LUGGAGE and t.hits >= getattr(tracker, 'min_hits', 1)]

        objects = persons + bags
        status = find_abandoned(dict(boxes=torch.stack([t.box for t in objects]) if objects else torch.zeros(0, 4),
                                     labels=[t.label for t in objects]), margin)
        unattended_since = {i: s for i, s in unattended_since.items() if i in {b.id for b in bags}}  # bagajele pierdute de tracker

        frame_out = [dict(id=p.id, box=p.box, text='#{}'.format(p.id), color=MONITOR_COLORS['persoana']) for p in persons]
        frame_timer = 0.0
        for bag, s in zip(bags, status[len(persons):]):
            # TODO: dacă bagajul este însoțit, ștergeți-l din unattended_since. Altfel, rețineți cadrul din care este
            # nesupravegheat (dacă nu există deja), calculați elapsed (în secunde) și actualizați frame_timer.
            # state: 'insotit', 'nesupravegheat' sau 'abandonat' (după alarm_after secunde)
            # text: eticheta afișată, de exemplu 'bagaj #3' sau 'ALARMA bagaj #3 4.2s' (fără diacritice: cv2.putText nu le poate desena)

            frame_out.append(dict(id=bag.id, box=bag.box, text=text, color=MONITOR_COLORS[state]))
        outputs.append(frame_out)
        timers.append(frame_timer)
    return outputs, np.array(timers)


monitor_3, timer_3 = monitor_luggage(MotionTracker(max_age=15), bag_dets, BAG_FPS)
alarm = timer_3 >= ALARM_AFTER
print('Prima alarmă la secunda {:.1f}; alarma activă în {:.0f}% din cadrele următoare'.format(
    frame_time(np.argmax(alarm)), 100 * alarm[np.argmax(alarm):].mean()))

In [ ]:
out_bag = render_video(VIDEO_3, os.path.join(OUT_DIR, 'abandoned-bag_monitor.mp4'), [monitor_3], draw_fn=draw_tracks, start=BAG_START, step=BAG_STEP)
display(Video(out_bag, embed=True, width=640))
frame_ids = [int((t * info_3['fps'] - BAG_START) / BAG_STEP) for t in (44, 48.5, 52, 68)]
show_images([draw_tracks(frame, monitor_3[i]) for i, frame in enumerate(read_frames(VIDEO_3, max(frame_ids) + 1, BAG_START, BAG_STEP)) if i in frame_ids],
            ['secunda {:.1f}'.format(frame_time(i)) for i in frame_ids], ncols=2, width=7)

- Cât timp proprietarii stau lângă rucsac, acesta este supravegheat (verde).
- După ce pleacă, cronometrul pornește (portocaliu), iar după `T` secunde rucsacul devine abandonat (roșu). Alarma rămâne activă până la sfârșitul videoclipului, deoarece nimeni nu se mai apropie de rucsac.
- Cât timp este purtat în mână, rucsacul își schimbă de mai multe ori ID-ul: este mic, parțial acoperit de corp și se mișcă odată cu persoana, deci este detectat intermitent. Pentru aplicația noastră acest lucru nu contează, pentru că un bagaj purtat este, prin definiție, supravegheat.

De ce contează tracker-ul? Faster R-CNN detectează rucsacul de pe jos în fiecare cadru, deci pe aceste detecții toți pașii produc aceeași alarmă. Simulăm un detector mai slab, ridicând pragul de încredere la 0,6: rucsacul (scor aproximativ 0,65) dispare acum în unele cadre. Comparăm cronometrul cu tracker-ul de la pasul 1 și cu cel de la pasul 3:

In [ ]:
weak_dets = [filter_detections(d, 0.6) for d in bag_dets]
times = [frame_time(f) for f in range(len(bag_dets))]
plt.figure(figsize=(12, 3.5))
for name, tracker in [('Pasul 1: IoU greedy', IoUTracker()), ('Pasul 3: SORT simplificat', MotionTracker(max_age=15))]:
    _, timer = monitor_luggage(tracker, weak_dets, BAG_FPS)
    after = timer[np.array(times) >= 50.5] >= ALARM_AFTER
    plt.plot(times, timer, label='{} (alarma activă în {:.0f}% din cadre după secunda 50,5)'.format(name, 100 * after.mean()))
plt.axhline(ALARM_AFTER, color='red', linestyle='--', label='prag de alarmă (T = {:.0f} s)'.format(ALARM_AFTER))
plt.xlabel('timp în videoclip (secunde)')
plt.ylabel('timp nesupravegheat (secunde)')
plt.legend(loc='upper left')
plt.show()

Cu pasul 1, fiecare cadru în care rucsacul nu este detectat șterge traiectoria; la următoarea detecție, rucsacul primește un ID nou și cronometrul pornește de la 0. Alarma se aprinde și se stinge repetat și este activă doar într-o parte din timp, deși rucsacul nu a fost atins. Cu pasul 3, traiectoria supraviețuiește golurilor de până la `max_age = 15` cadre (o secundă), iar cronometrul continuă neîntrerupt.

Dacă ridicăm pragul și mai mult (încercați 0,65 sau 0,7), golurile devin mai lungi decât `max_age` și și pasul 3 începe să piardă rucsacul: tracker-ul compensează erorile scurte ale detectorului, dar nu poate înlocui un detector bun.

# #TODO:
1. Implementați părțile marcate cu `TODO` din secțiunea 4: `match_greedy` (4.1), `LifecycleTracker.update()` (4.2), `MotionTracker.update()` (4.3) și `monitor_luggage` (4.4). Rulați celulele în ordine: fiecare pas al tracker-ului folosește funcțiile din pașii anteriori.
2. Bagajele abandonate: ce se întâmplă dacă două bagaje se ating între ele, iar unul dintre ele se intersectează cu o persoană? Ar trebui al doilea bagaj să fie considerat însoțit? Propuneți o regulă și implementați-o.
3. Regula de posesie (secțiunea 2.3) lucrează în planul imaginii. Ce informație suplimentară ar fi necesară pentru a măsura distanța reală dintre bagaj și persoană? Cum ați putea aproxima această distanță folosind doar casetele (indiciu: poziția bazei casetei în imagine și înălțimea persoanei)?
4. Extindeți aplicația 1 la video: un bagaj declanșează alarma doar dacă rămâne abandonat în cel puțin `T` secunde consecutive. Ce problemă apare dacă detectorul ratează bagajul într-un singur cadru? Cum o rezolvați?
5. Rulați detecția pe video cu `BATCH_SIZE` egal cu 1, 4 și 16 și comparați vitezele. De ce crește viteza cu dimensiunea batch-ului? De ce nu crește nelimitat?
6. Variați `max_age` (1, 5, 10, 30) și `iou_threshold` (0,1, 0,3, 0,5) în `MotionTracker` și urmăriți numărul de ID-uri pe `vtest.avi`. De ce o valoare foarte mare a lui `max_age` poate produce schimbări de ID (două persoane care își "fură" traiectoriile)?